## Assignment 51 — Week 5, Day 1
- Implement linear regression in Python with scikit-learn.
Reference: https://stackabuse.com/linear-regression-in-python-with-scikit-learn/


In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


X = np.array([[1], [2], [3], [4], [5],
              [6], [7], [8], [9], [10]])

y = np.array([15, 20, 25, 30, 35, 40, 45, 50, 55, 60])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


model = LinearRegression()


model.fit(X_train, y_train)


predictions = model.predict(X_test)


print("Actual values:", y_test)
print("Predicted values:", predictions)
print("Slope:", model.coef_[0])
print("Intercept:", model.intercept_)
print("Mean Absolute Error:", mean_absolute_error(y_test, predictions))


new_prediction = model.predict(np.array([[11]]))
print("Prediction for 11 hours:", new_prediction[0])

Actual values: [55 20]
Predicted values: [55. 20.]
Slope: 5.000000000000001
Intercept: 9.999999999999996
Mean Absolute Error: 0.0
Prediction for 11 hours: 65.0


## Assignment 52 — Week 5, Day 3
- Perform logistic regression.
Reference: https://towardsdatascience.com/logistic-regression-using-python-sklearn-numpy-mnist-handwriting-recognition-matplotlib-a6b31e2b166a
https://www.datacamp.com/community/tutorials/understanding-logistic-regression-python


In [ ]:


import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/ammarmahmood1999/HeartHealthPrediction/master/heart.csv"

df = pd.read_csv(URL)
print(df.shape)
print(df.head())

X = df.drop(columns=["Target"])
y = df["Target"]

X = X.drop_duplicates()
y = y[X.index]

# 4. Split  - before anything that computes a statistic
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = X.select_dtypes("object").columns.tolist()  #

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])


models = {
    "LogReg": (LogisticRegression(max_iter=2000), {"model__C": [0.01, 0.1, 1, 10]}),
    "KNN":    (KNeighborsClassifier(), {"model__n_neighbors": [3, 5, 7, 9, 11]}),
    "RF":     (RandomForestClassifier(random_state=42),
               {"model__n_estimators": [200], "model__max_depth": [None, 4, 6]}),
}


best_name, best_score, best_model = None, 0, None
for name, (clf, grid) in models.items():
    gs = GridSearchCV(Pipeline([("prep", prep), ("model", clf)]), grid, cv=5)
    gs.fit(X_train, y_train)
    print(f"{name:8} cv={gs.best_score_:.4f}  {gs.best_params_}")
    if gs.best_score_ > best_score:
        best_name, best_score, best_model = name, gs.best_score_, gs.best_estimator_


print(f"\nbest: {best_name} ({best_score:.4f})")

y_pred = best_model.predict(X_test)
print(f"test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
report = classification_report(y_test, y_pred, target_names=["no disease", "disease"])
print("\n".join(l for l in report.split("\n") if "avg" not in l))

(303, 14)
   Age  Sex     ChestPain  RestBP  Chol  Fbs  RestECG  MaxHR  ExAng  Oldpeak  \
0   63    1       typical     145   233    1        2    150      0      2.3   
1   67    1  asymptomatic     160   286    0        2    108      1      1.5   
2   67    1  asymptomatic     120   229    0        2    129      1      2.6   
3   37    1    nonanginal     130   250    0        0    187      0      3.5   
4   41    0    nontypical     130   204    0        2    172      0      1.4   

   Slope  Ca        Thal  Target  
0      3   0       fixed       0  
1      2   3      normal       1  
2      2   2  reversable       1  
3      3   0      normal       0  
4      1   0      normal       0  


C:\Users\Hi\AppData\Local\Temp\ipykernel_10488\355696568.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns.tolist()  #


LogReg   cv=0.8345  {'model__C': 10}
KNN      cv=0.8096  {'model__n_neighbors': 7}
RF       cv=0.8179  {'model__max_depth': 4, 'model__n_estimators': 200}

best: LogReg (0.8345)
test accuracy: 0.9016
[[29  4]
 [ 2 26]]
              precision    recall  f1-score   support

  no disease       0.94      0.88      0.91        33
     disease       0.87      0.93      0.90        28

    accuracy                           0.90        61

